In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\jaina\OneDrive\Desktop\Projects\Mkisans\Demand Analytics & Forecasting System\data\raw\mp_data.csv")
print(df.shape)
df.head()

(2394341, 15)


,State Name,District Name,Market Name,Variety,Group,Arrivals (Tonnes),Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal),Reported Date,Commodity_File,Arrivals,Min Price,Max Price,Modal Price
0,Madhya Pradesh,Bhind,Lahar,Mill Quality,Cereals,1.4,2400.0,2400.0,2400.0,31 Jan 2024,Wheat,NaN,NaN,NaN,NaN
1,Madhya Pradesh,Bhind,Lahar,Mill Quality,Cereals,8.0,2510.0,2525.0,2510.0,25 Jan 2024,Wheat,NaN,NaN,NaN,NaN
2,Madhya Pradesh,Bhind,Lahar,Mill Quality,Cereals,1.5,2590.0,2590.0,2590.0,14 Jan 2024,Wheat,NaN,NaN,NaN,NaN
3,Madhya Pradesh,Bhind,Lahar,Other,Cereals,65.7,2315.0,2406.0,2380.0,20 Dec 2023,Wheat,NaN,NaN,NaN,NaN
4,Madhya Pradesh,Bhind,Lahar,Other,Cereals,86.4,2300.0,2391.0,2350.0,19 Dec 2023,Wheat,NaN,NaN,NaN,NaN


In [3]:
df.isnull().sum()

State Name                         0
District Name                      0
Market Name                        0
Variety                            0
Group                              0
Arrivals (Tonnes)             308792
Min Price (Rs./Quintal)       308792
Max Price (Rs./Quintal)       308793
Modal Price (Rs./Quintal)     308793
Reported Date                      0
Commodity_File                     0
Arrivals                     2085549
Min Price                    2085549
Max Price                    2085549
Modal Price                  2085549
dtype: int64

In [4]:

df = df[[
    'State Name',
    'District Name',
    'Market Name',
    'Variety',
    'Group',
    'Arrivals (Tonnes)',
    'Min Price (Rs./Quintal)',
    'Max Price (Rs./Quintal)',
    'Modal Price (Rs./Quintal)',
    'Reported Date',
    'Commodity_File'
]]

In [5]:
df = df.rename(columns={
    'State Name': 'state',
    'District Name': 'district',
    'Market Name': 'market',
    'Variety': 'variety',
    'Group': 'group',
    'Arrivals (Tonnes)': 'arrivals',
    'Min Price (Rs./Quintal)': 'min_price',
    'Max Price (Rs./Quintal)': 'max_price',
    'Modal Price (Rs./Quintal)': 'modal_price',
    'Reported Date': 'date',
    'Commodity_File': 'commodity'
})

In [6]:
df.isnull().sum()

state               0
district            0
market              0
variety             0
group               0
arrivals       308792
min_price      308792
max_price      308793
modal_price    308793
date                0
commodity           0
dtype: int64

In [7]:
df = df.dropna(subset=[
    'modal_price',
    'min_price',
    'max_price'
])

In [8]:
print(df.shape)

(2085548, 11)


In [9]:
df['date'] = pd.to_datetime(
    df['date'],
    format='mixed',
    dayfirst=True)

In [10]:
df = df.sort_values('date')

In [11]:
print(df['date'].min())
print(df['date'].max())

2001-03-27 00:00:00
2024-02-02 00:00:00


In [12]:
numeric_cols = [
    "min_price",
    "max_price",
    "modal_price",
    "arrivals"
]

for col in numeric_cols:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [13]:
df = df[
    df["modal_price"] > 0
]

df = df[
    df["arrivals"] >= 0
]

In [14]:
swap = df["min_price"] > df["max_price"]

df.loc[
    swap,
    ["min_price", "max_price"]
] = df.loc[
    swap,
    ["max_price", "min_price"]
].values

In [15]:
df["modal_price"] = df["modal_price"].clip(
    lower=df["min_price"],
    upper=df["max_price"]
)

In [16]:
df["market_key"] = (

    df["state"]
    .astype(str)
    .str.strip()
    .str.title()

    + "_"

    + df["district"]
    .astype(str)
    .str.strip()
    .str.title()

    + "_"

    + df["market"]
    .astype(str)
    .str.strip()
    .str.title()

    + "_"

    + df["commodity"]
    .astype(str)
    .str.strip()
    .str.title()
)

In [17]:
df = df.sort_values(
    ["market_key", "date"]
).reset_index(drop=True)

In [18]:
df = df[
    df["commodity"] == "Soybean"
].copy()

In [19]:
LAG_DAYS = [1, 3, 7, 14, 30]

ROLL_WINDOWS = [7, 14, 30]


def add_features_for_group(group):

    group = (
        group
        .set_index("date")
        .sort_index()
    )

    # Daily calendar
    price_daily = (
        group["modal_price"]
        .resample("D")
        .mean()
    )

    arrival_daily = (
        group["arrivals"]
        .resample("D")
        .mean()
    )

    # ---------- PRICE LAGS ----------
    for lag in LAG_DAYS:

        group[f"price_lag_{lag}d"] = (
            group.index.map(

                lambda d:
                price_daily.get(
                    d - pd.Timedelta(days=lag)
                )
            )
        )

    # ---------- ARRIVAL LAGS ----------
    for lag in [1, 7, 14]:

        group[f"arrival_lag_{lag}d"] = (
            group.index.map(

                lambda d:
                arrival_daily.get(
                    d - pd.Timedelta(days=lag)
                )
            )
        )

    # ---------- ROLLING FEATURES ----------
    for w in ROLL_WINDOWS:

        roll = price_daily.rolling(
            w,
            min_periods=max(3, w // 3)
        )

        group[f"rolling_mean_{w}d"] = (
            group.index.map(
                roll.mean()
            )
        )

        group[f"rolling_std_{w}d"] = (
            group.index.map(
                roll.std()
            )
        )

    # ---------- MOMENTUM ----------
    lag7 = group["price_lag_7d"]

    group["price_momentum_7d"] = (

        (group["modal_price"] - lag7)

        /

        (lag7 + 1e-8)
    )

    # ---------- ARRIVAL/PRICE ----------
    group["arrival_price_ratio"] = (

        group["arrivals"]

        /

        (group["modal_price"] + 1)
    )

    return group.reset_index()

In [20]:
df = (
    df
    .groupby(
        "market_key",
        group_keys=False
    )
    .apply(add_features_for_group)
    .reset_index(drop=True)
)

C:\Users\jaina\AppData\Local\Temp\ipykernel_4024\3253981255.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_features_for_group)


In [21]:
df["day"] = df["date"].dt.day

df["month"] = df["date"].dt.month

df["week"] = (
    df["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

df["quarter"] = df["date"].dt.quarter

df["year"] = df["date"].dt.year

In [22]:
df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

In [23]:
df["target"] = (

    df
    .groupby("market_key")

    ["modal_price"]

    .shift(-7)
)

In [24]:
df = df[
    df["target"].notna()
]

In [26]:
# FIX ROLLING FEATURES

for w in ROLL_WINDOWS:

    rolling_mean = (
        df
        .groupby("market_key")["modal_price"]
        .transform(
            lambda s: s.rolling(
                w,
                min_periods=max(3, w // 3)
            ).mean()
        )
    )

    rolling_std = (
        df
        .groupby("market_key")["modal_price"]
        .transform(
            lambda s: s.rolling(
                w,
                min_periods=max(3, w // 3)
            ).std()
        )
    )

    df[f"rolling_mean_{w}d"] = rolling_mean
    df[f"rolling_std_{w}d"] = rolling_std

In [27]:
# FIX PRICE LAGS

for lag in LAG_DAYS:

    df[f"price_lag_{lag}d"] = (
        df
        .groupby("market_key")["modal_price"]
        .shift(lag)
    )

In [28]:
# FIX ARRIVAL LAGS

for lag in [1, 7, 14]:

    df[f"arrival_lag_{lag}d"] = (
        df
        .groupby("market_key")["arrivals"]
        .shift(lag)
    )

In [29]:
df["price_momentum_7d"] = (
    (df["modal_price"] - df["price_lag_7d"])
    /
    (df["price_lag_7d"] + 1e-8)
)

In [30]:
df[cols].head(20)

,date,modal_price,price_lag_1d,price_lag_7d,rolling_mean_7d,price_momentum_7d,target


In [31]:
cols = [

    "date",
    "modal_price",

    "price_lag_1d",
    "price_lag_7d",

    "rolling_mean_7d",

    "price_momentum_7d",

    "target"
]

df[cols].head(20)

,date,modal_price,price_lag_1d,price_lag_7d,rolling_mean_7d,price_momentum_7d,target


In [32]:
df.isnull().sum().sort_values(
    ascending=False
).head(20)

state          0
district       0
market         0
variety        0
group          0
arrivals       0
min_price      0
max_price      0
modal_price    0
date           0
commodity      0
market_key     0
day            0
month          0
week           0
quarter        0
year           0
month_sin      0
month_cos      0
target         0
dtype: int64

In [34]:
df.to_parquet(

    r"C:\Users\jaina\OneDrive\Desktop\Projects\Mkisans\Demand Analytics & Forecasting System\data\processed/final_features_df.parquet",

    index=False
)